In [169]:
import warnings
from causalnex.structure import StructureModel
warnings.filterwarnings("ignore")  # silence warnings
from causalnex.plots import plot_structure, NODE_STYLE, EDGE_STYLE
from causalnex.network import BayesianNetwork
import pandas as pd
from causalnex.discretiser import Discretiser
import numpy as np
from sklearn.model_selection import train_test_split
from causalnex.structure.notears import from_pandas_lasso
from IPython.display import Image
from causalnex.evaluation import classification_report
from causalnex.evaluation import roc_auc
from causalnex.inference import InferenceEngine
from causalnex.evaluation import classification_report
import copy
import networkx as nx
import turtorial_utils as utils

In [170]:
df_fi = pd.read_csv("/Users/vladasverkelis/Documents/Doktorantūra/Straipsnis_1/Context_influance_to_EU_structural_funds/Data/y/fi_bn_no_context.csv")

In [171]:
df_fi = df_fi.drop(["Unnamed: 0"], axis = 1)

In [172]:
df_fi

,GDPC,VABI,EMP,QOA,QOR,YUA,REST,EBSG,GGFC,LBGD,FDI,PC,ARP,CO2C,PAM,EUPC
0,35369.427763,30.67,73.892857,6.310000,5.64,16.43,29.561,4.794744,20.768314,5.292982,8.57,4.4,13.0,0.000013,341.020794,6.033898
1,36559.083931,29.57,73.892857,6.430000,5.84,16.34,31.071,3.615903,21.566205,4.569814,6.79,4.9,13.6,0.000011,338.794727,9.016756
2,34039.968375,25.66,72.600000,6.330000,5.89,21.38,31.045,2.038683,24.069112,-1.860401,-3.48,4.7,13.8,0.000011,338.202247,34.478428
3,35080.114078,26.19,71.900000,6.150000,5.86,21.29,32.166,1.364890,23.685735,-2.369424,4.90,4.4,13.1,0.000012,322.947761,27.559638
4,36682.446617,25.04,72.700000,6.160000,5.83,19.93,32.532,-0.781159,23.401887,-0.488730,-2.18,4.8,13.7,0.000011,306.122449,44.801132
5,37011.280629,23.36,73.000000,6.200000,6.09,18.87,34.222,-1.560551,24.176307,-1.104413,1.92,4.7,13.2,0.000010,313.863216,40.733725
6,37414.607025,23.30,72.500000,6.220000,6.10,19.82,36.630,-0.984781,24.635744,-1.825088,-1.83,4.5,11.8,0.000010,293.382353,61.528149
7,37691.943183,23.06,72.200000,6.170000,5.87,20.45,38.633,-1.015764,24.631415,-2.267616,6.42,3.8,12.8,0.000009,259.890110,39.404991
8,38359.487336,23.30,71.800000,6.050000,5.78,22.30,39.230,-0.398683,24.522817,-1.810725,7.54,2.8,12.4,0.000008,235.218978,16.873910
9,39254.796323,23.48,72.400000,6.200000,5.70,20.06,38.943,-1.055550,23.868773,-1.298460,2.19,2.9,11.6,0.000009,229.090909,18.148198


In [173]:
fi = pd.DataFrame()
fi = df_fi.copy()
# fi['EUPC_3'] = fi['EUPC'].shift(3)
# fi['GGFC_3'] = fi['GGFC'].shift(3)
# fi['QOR_2'] = fi['QOR'].shift(2)
# fi['CO2C_1'] = fi['CO2C'].shift(1)

In [174]:
fi = fi.dropna()

In [175]:
sm = StructureModel()

In [176]:
sm.add_edges_from([
    ('EUPC', 'YUA'),
     ('LBGD', 'YUA'),
    ('QOA', 'LBGD'),
     ('GGFC', 'LBGD'),
    ('QOR', 'QOA'),
    ('CO2C', 'QOA'),
    ('CO2C', 'GGFC')
])

In [177]:
sm.edges

OutEdgeView([('EUPC', 'YUA'), ('LBGD', 'YUA'), ('QOA', 'LBGD'), ('GGFC', 'LBGD'), ('QOR', 'QOA'), ('CO2C', 'QOA'), ('CO2C', 'GGFC')])

In [178]:
viz = plot_structure(
    sm,
    all_node_attributes=NODE_STYLE.WEAK,
    all_edge_attributes=EDGE_STYLE.WEAK,
)

viz.toggle_physics(False)
viz.show("Graphs/fully_connected_fi_1.html")

Graphs/fully_connected_fi_1.html


In [179]:
bn = BayesianNetwork(sm)

In [180]:
# Check how much data is left
print(f"Data rows remaining after lag: {len(fi)}")

# 2. Information-Preserving Discretization (Quantiles)
# Try 3 buckets (Low, Mid, High) if you have >30 rows. 
# If <30 rows, stick to 2 buckets to avoid empty bin errors.
num_buckets = 3 if len(fi) > 30 else 2 
discretised_fi = fi.copy()

for col in discretised_fi.columns:
    try:
        # 'quantile' preserves distribution shape better than 'fixed'
        discretised_fi[col] = Discretiser(
            method="quantile", 
            num_buckets=num_buckets
        ).transform(discretised_fi[col].values)
    except ValueError:
        # Fallback for columns with very low variance (like CO2C might be)
        print(f"Column {col} has low variance, falling back to median split.")
        split = discretised_fi[col].median()
        discretised_fi[col] = Discretiser(method="fixed", numeric_split_points=[split]).transform(discretised_fi[col].values)

    # Map integers to strings
    mapper = {0: "Low", 1: "Medium", 2: "High"} if num_buckets == 3 else {0: "Low", 1: "High"}
    discretised_fi[col] = discretised_fi[col].map(mapper)

# 3. Fit the Network
bn.fit_node_states(discretised_fi)

Data rows remaining after lag: 16
Column GDPC has low variance, falling back to median split.
Column VABI has low variance, falling back to median split.
Column EMP has low variance, falling back to median split.
Column QOA has low variance, falling back to median split.
Column QOR has low variance, falling back to median split.
Column YUA has low variance, falling back to median split.
Column REST has low variance, falling back to median split.
Column EBSG has low variance, falling back to median split.
Column GGFC has low variance, falling back to median split.
Column LBGD has low variance, falling back to median split.
Column FDI has low variance, falling back to median split.
Column PC has low variance, falling back to median split.
Column ARP has low variance, falling back to median split.
Column CO2C has low variance, falling back to median split.
Column PAM has low variance, falling back to median split.
Column EUPC has low variance, falling back to median split.


In [181]:
# discretised_fi = discretised_fi.drop(["REST", "GGFC_3"], axis = 1)


In [182]:
discretised_fi = discretised_fi.reset_index(drop=True)
bn.fit_node_states(discretised_fi)
baseline_auc = utils.get_avg_auc_all_info(discretised_fi, bn)
print(f"Baseline AUC: {baseline_auc}")

Processing fold 0 using 7 cores takes 5.244958877563477 seconds
Processing fold 1 using 7 cores takes 5.024356126785278 seconds


ValueError: unknown format is not supported

In [ ]:
edges_to_add = [('LV', 'EUPC_3'), ('LV', 'YUA')]
edges_to_remove = [('EUPC_3', 'YUA')]

bn_with_lv = copy.deepcopy(bn)
bn_with_lv.add_node(
    "LV",
    edges_to_add=edges_to_add,
    edges_to_remove=edges_to_remove,
)

In [ ]:
viz = utils.plot_pretty_structure(bn_with_lv.structure, edges_to_highlight=edges_to_add)
viz.show("Graphs/node_added_fi_1.html")

In [ ]:
discretised_fi['LV'] = None
lv_states = [0, 1, 2, 3, 4]

proposed_auc = utils.get_avg_auc_lvs(discretised_fi, bn_with_lv, lv_states)
print(f"AUC from adding LV between 'EUPC_3' and 'YUA': {proposed_auc}")

In [87]:
import warnings
from causalnex.structure import StructureModel
warnings.filterwarnings("ignore")  # silence warnings
from causalnex.structure.notears import from_pandas
from causalnex.plots import plot_structure, NODE_STYLE, EDGE_STYLE
from causalnex.network import BayesianNetwork
import pandas as pd
from causalnex.discretiser import Discretiser
import numpy as np
from sklearn.model_selection import train_test_split
from causalnex.structure.notears import from_pandas_lasso
from IPython.display import Image
from causalnex.evaluation import classification_report
from causalnex.evaluation import roc_auc
from causalnex.inference import InferenceEngine
from causalnex.evaluation import classification_report
import copy
import networkx as nx
import turtorial_utils as utils

In [88]:
df_fi_1 = pd.read_csv("/Users/vladasverkelis/Documents/Doktorantūra/Straipsnis_1/Context_influance_to_EU_structural_funds/Data/y/fi_bn_no_context.csv")

In [89]:
df_fi_1 = df_fi_1.drop(["Unnamed: 0"], axis = 1)

In [90]:
df_fi_1

,GDPC,VABI,EMP,QOA,QOR,YUA,REST,EBSG,GGFC,LBGD,FDI,PC,ARP,CO2C,PAM,EUPC
0,35369.427763,30.67,73.892857,6.310000,5.64,16.43,29.561,4.794744,20.768314,5.292982,8.57,4.4,13.0,0.000013,341.020794,6.033898
1,36559.083931,29.57,73.892857,6.430000,5.84,16.34,31.071,3.615903,21.566205,4.569814,6.79,4.9,13.6,0.000011,338.794727,9.016756
2,34039.968375,25.66,72.600000,6.330000,5.89,21.38,31.045,2.038683,24.069112,-1.860401,-3.48,4.7,13.8,0.000011,338.202247,34.478428
3,35080.114078,26.19,71.900000,6.150000,5.86,21.29,32.166,1.364890,23.685735,-2.369424,4.90,4.4,13.1,0.000012,322.947761,27.559638
4,36682.446617,25.04,72.700000,6.160000,5.83,19.93,32.532,-0.781159,23.401887,-0.488730,-2.18,4.8,13.7,0.000011,306.122449,44.801132
5,37011.280629,23.36,73.000000,6.200000,6.09,18.87,34.222,-1.560551,24.176307,-1.104413,1.92,4.7,13.2,0.000010,313.863216,40.733725
6,37414.607025,23.30,72.500000,6.220000,6.10,19.82,36.630,-0.984781,24.635744,-1.825088,-1.83,4.5,11.8,0.000010,293.382353,61.528149
7,37691.943183,23.06,72.200000,6.170000,5.87,20.45,38.633,-1.015764,24.631415,-2.267616,6.42,3.8,12.8,0.000009,259.890110,39.404991
8,38359.487336,23.30,71.800000,6.050000,5.78,22.30,39.230,-0.398683,24.522817,-1.810725,7.54,2.8,12.4,0.000008,235.218978,16.873910
9,39254.796323,23.48,72.400000,6.200000,5.70,20.06,38.943,-1.055550,23.868773,-1.298460,2.19,2.9,11.6,0.000009,229.090909,18.148198


In [91]:
fi_1 = pd.DataFrame()
fi_1 = df_fi_1.copy()
fi_1['EUPC_2'] = fi_1['EUPC'].shift(2)

In [92]:
fi_1 = fi_1.dropna()

In [93]:
sm1 = StructureModel()

In [94]:
sm1.add_edges_from([
    ('EUPC_2', 'FDI'),
    ('EUPC_2', 'ARP'),
#      ('ARP', 'EUPC')
    
])

In [95]:
sm1.edges

OutEdgeView([('EUPC_2', 'FDI'), ('EUPC_2', 'ARP')])

In [96]:
viz = plot_structure(
    sm1,
    all_node_attributes=NODE_STYLE.WEAK,
    all_edge_attributes=EDGE_STYLE.WEAK,
)

viz.toggle_physics(False)
viz.show("Graphs/fully_connected_fi_2.html")

Graphs/fully_connected_fi_2.html


In [97]:
bn1 = BayesianNetwork(sm1)

In [103]:
discretised_fi = pd.DataFrame(index=fi_1.index)

for col in fi_1.columns:
    no_unique = fi_1[col].nunique()
    
    if no_unique <= 1:
        discretised_fi[col] = 0
    else:
        try:
            discretised_fi[col] = pd.qcut(
                fi_1[col], 
                q=min(3, no_unique),  
                labels=False,
                duplicates='drop'
            ).astype(int)
        except ValueError:
            discretised_fi[col] = fi_1[col].rank(method='dense').astype(int) - 1

print("Discretised data:")
print(discretised_fi)

Discretised data:
    GDPC  VABI  EMP  QOA  QOR  YUA  REST  EBSG  GGFC  LBGD  FDI  PC  ARP  \
2      0     2    1    2    2    2     0     2     1     0    0   2    2   
3      0     2    0    0    2    2     0     2     0     0    1   2    2   
4      0     2    1    0    1    1     0     1     0     2    0   2    2   
5      0     0    1    0    2    0     0     0     1     1    1   2    2   
6      0     0    0    1    2    1     0     0     2     1    0   2    0   
7      1     0    0    0    2    2     1     0     2     0    2   1    2   
8      1     0    0    0    1    2     1     1     2     1    2   1    1   
9      1     0    0    0    1    1     1     0     1     1    1   1    0   
10     1     2    1    2    1    1     1     2     0     2    2   0    0   
11     2     1    2    0    0    0     2     1     0     2    0   0    1   
12     2     1    2    2    0    0     2     2     0     2    2   0    0   
13     2     1    2    1    0    2     2     2     2     0    0   0   

In [104]:
discretised_fi = discretised_fi.reset_index(drop=True)
bn1.fit_node_states(discretised_fi)
baseline_auc = utils.get_avg_auc_all_info(discretised_fi, bn1)
print(f"Baseline AUC: {baseline_auc}")

Processing fold 0 using 7 cores takes 5.261120796203613 seconds
Processing fold 1 using 7 cores takes 4.827132225036621 seconds
Processing fold 2 using 7 cores takes 4.889833927154541 seconds
Processing fold 3 using 7 cores takes 4.662934064865112 seconds
Processing fold 4 using 7 cores takes 4.701222896575928 seconds
Baseline AUC: 0.28148148148148144


In [105]:
edges_to_add = [('LV', 'EUPC_2'), ('LV', 'FDI')]
edges_to_remove = [('EUPC_2', 'FDI')]

bn_with_lv = copy.deepcopy(bn1)
bn_with_lv.add_node(
    "LV",
    edges_to_add=edges_to_add,
    edges_to_remove=edges_to_remove,
)

In [106]:
viz = utils.plot_pretty_structure(bn_with_lv.structure, edges_to_highlight=edges_to_add)
viz.show("Graphs/node_added_FI_3.html")

Graphs/node_added_FI_3.html


In [107]:
discretised_fi['LV'] = None
lv_states = [0, 1, 2, 3, 4]

proposed_auc = utils.get_avg_auc_lvs(discretised_fi, bn_with_lv, lv_states)
print(f"AUC from adding LV between 'EUPC_2' and 'FDI': {proposed_auc}")

Processing fold 0 using 7 cores takes 5.435314178466797 seconds
Processing fold 1 using 7 cores takes 4.938570022583008 seconds
Processing fold 2 using 7 cores takes 4.840914964675903 seconds
Processing fold 3 using 7 cores takes 4.765208721160889 seconds
Processing fold 4 using 7 cores takes 4.702479839324951 seconds
AUC from adding LV between 'EUPC_2' and 'FDI': 0.37268518518518523


In [108]:
edges_to_add = [('LV', 'EUPC_2'), ('LV', 'ARP')]
edges_to_remove = [('EUPC_2', 'ARP')]

bn_with_lv = copy.deepcopy(bn1)
bn_with_lv.add_node(
    "LV",
    edges_to_add=edges_to_add,
    edges_to_remove=edges_to_remove,
)

In [109]:
viz = utils.plot_pretty_structure(bn_with_lv.structure, edges_to_highlight=edges_to_add)
viz.show("Graphs/node_added_FI_4.html")

Graphs/node_added_FI_4.html


In [110]:
discretised_fi['LV'] = None
lv_states = [0, 1, 2, 3, 4]

proposed_auc = utils.get_avg_auc_lvs(discretised_fi, bn_with_lv, lv_states)
print(f"AUC from adding LV between 'EUPC_2' and 'ARP': {proposed_auc}")

Processing fold 0 using 7 cores takes 5.422722101211548 seconds
Processing fold 1 using 7 cores takes 4.9407103061676025 seconds
Processing fold 2 using 7 cores takes 5.236337900161743 seconds
Processing fold 3 using 7 cores takes 5.459999084472656 seconds
Processing fold 4 using 7 cores takes 4.82905912399292 seconds
AUC from adding LV between 'EUPC_2' and 'ARP': 0.47037037037037044


In [128]:
import warnings
from causalnex.structure import StructureModel
warnings.filterwarnings("ignore")  # silence warnings
from causalnex.structure.notears import from_pandas
from causalnex.plots import plot_structure, NODE_STYLE, EDGE_STYLE
from causalnex.network import BayesianNetwork
import pandas as pd
from causalnex.discretiser import Discretiser
import numpy as np
from sklearn.model_selection import train_test_split
from causalnex.structure.notears import from_pandas_lasso
from IPython.display import Image
from causalnex.evaluation import classification_report
from causalnex.evaluation import roc_auc
from causalnex.inference import InferenceEngine
from causalnex.evaluation import classification_report
import copy
import networkx as nx
import turtorial_utils as utils

In [129]:
df_fi_1 = pd.read_csv("/Users/vladasverkelis/Documents/Doktorantūra/Straipsnis_1/Context_influance_to_EU_structural_funds/Data/y/fi_bn_no_context.csv")

In [130]:
df_fi_1 = df_fi_1.drop(["Unnamed: 0"], axis = 1)

In [131]:
df_fi_1

,GDPC,VABI,EMP,QOA,QOR,YUA,REST,EBSG,GGFC,LBGD,FDI,PC,ARP,CO2C,PAM,EUPC
0,35369.427763,30.67,73.892857,6.310000,5.64,16.43,29.561,4.794744,20.768314,5.292982,8.57,4.4,13.0,0.000013,341.020794,6.033898
1,36559.083931,29.57,73.892857,6.430000,5.84,16.34,31.071,3.615903,21.566205,4.569814,6.79,4.9,13.6,0.000011,338.794727,9.016756
2,34039.968375,25.66,72.600000,6.330000,5.89,21.38,31.045,2.038683,24.069112,-1.860401,-3.48,4.7,13.8,0.000011,338.202247,34.478428
3,35080.114078,26.19,71.900000,6.150000,5.86,21.29,32.166,1.364890,23.685735,-2.369424,4.90,4.4,13.1,0.000012,322.947761,27.559638
4,36682.446617,25.04,72.700000,6.160000,5.83,19.93,32.532,-0.781159,23.401887,-0.488730,-2.18,4.8,13.7,0.000011,306.122449,44.801132
5,37011.280629,23.36,73.000000,6.200000,6.09,18.87,34.222,-1.560551,24.176307,-1.104413,1.92,4.7,13.2,0.000010,313.863216,40.733725
6,37414.607025,23.30,72.500000,6.220000,6.10,19.82,36.630,-0.984781,24.635744,-1.825088,-1.83,4.5,11.8,0.000010,293.382353,61.528149
7,37691.943183,23.06,72.200000,6.170000,5.87,20.45,38.633,-1.015764,24.631415,-2.267616,6.42,3.8,12.8,0.000009,259.890110,39.404991
8,38359.487336,23.30,71.800000,6.050000,5.78,22.30,39.230,-0.398683,24.522817,-1.810725,7.54,2.8,12.4,0.000008,235.218978,16.873910
9,39254.796323,23.48,72.400000,6.200000,5.70,20.06,38.943,-1.055550,23.868773,-1.298460,2.19,2.9,11.6,0.000009,229.090909,18.148198


In [132]:
fi_1 = pd.DataFrame()
fi_1 = df_fi_1.copy()

In [133]:
fi_1 = fi_1.dropna()

In [134]:
sm1 = StructureModel()

In [135]:
sm1.add_edges_from([
    ('ARP', 'EUPC'),
    
])

In [136]:
sm1.edges

OutEdgeView([('ARP', 'EUPC')])

In [137]:
viz = plot_structure(
    sm1,
    all_node_attributes=NODE_STYLE.WEAK,
    all_edge_attributes=EDGE_STYLE.WEAK,
)

viz.toggle_physics(False)
viz.show("Graphs/fully_connected_fi_2.html")

Graphs/fully_connected_fi_2.html


In [138]:
bn1 = BayesianNetwork(sm1)

In [139]:
discretised_fi = pd.DataFrame(index=fi_1.index)

for col in fi_1.columns:
    no_unique = fi_1[col].nunique()
    
    if no_unique <= 1:
        discretised_fi[col] = 0
    else:
        try:
            discretised_fi[col] = pd.qcut(
                fi_1[col], 
                q=min(3, no_unique),  
                labels=False,
                duplicates='drop'
            ).astype(int)
        except ValueError:
            discretised_fi[col] = fi_1[col].rank(method='dense').astype(int) - 1

print("Discretised data:")
print(discretised_fi)

Discretised data:
    GDPC  VABI  EMP  QOA  QOR  YUA  REST  EBSG  GGFC  LBGD  FDI  PC  ARP  \
0      0     2    1    2    1    0     0     2     0     2    2   1    1   
1      0     2    1    2    1    0     0     2     0     2    2   2    2   
2      0     2    0    2    2    2     0     2     1     0    0   2    2   
3      0     2    0    0    2    2     0     2     1     0    1   1    2   
4      0     1    1    0    1    1     0     1     0     1    0   2    2   
5      0     0    1    0    2    1     0     0     1     1    0   2    2   
6      1     0    0    1    2    1     1     0     2     0    0   2    0   
7      1     0    0    0    2    2     1     0     2     0    1   1    1   
8      1     0    0    0    1    2     1     1     2     1    2   0    1   
9      1     0    0    0    1    1     1     0     1     1    1   1    0   
10     1     1    1    2    0    1     1     1     0     1    2   0    0   
11     2     0    2    0    0    0     2     0     0     2    0   0   

In [140]:
discretised_fi = discretised_fi.reset_index(drop=True)
bn1.fit_node_states(discretised_fi)
baseline_auc = utils.get_avg_auc_all_info(discretised_fi, bn1)
print(f"Baseline AUC: {baseline_auc}")

Processing fold 0 using 7 cores takes 5.812650918960571 seconds
Processing fold 1 using 7 cores takes 4.576502084732056 seconds
Processing fold 2 using 7 cores takes 4.296717882156372 seconds
Processing fold 3 using 7 cores takes 4.237874984741211 seconds
Processing fold 4 using 7 cores takes 4.194668769836426 seconds
Baseline AUC: 0.26510416666666664


In [141]:
edges_to_add = [('LV', 'ARP'), ('LV', 'EUPC')]
edges_to_remove = [('ARP', 'EUPC')]

bn_with_lv = copy.deepcopy(bn1)
bn_with_lv.add_node(
    "LV",
    edges_to_add=edges_to_add,
    edges_to_remove=edges_to_remove,
)

In [142]:
viz = utils.plot_pretty_structure(bn_with_lv.structure, edges_to_highlight=edges_to_add)
viz.show("Graphs/node_added_FI_4.html")

Graphs/node_added_FI_4.html


In [143]:
discretised_fi['LV'] = None
lv_states = [0, 1, 2, 3, 4]

proposed_auc = utils.get_avg_auc_lvs(discretised_fi, bn_with_lv, lv_states)
print(f"AUC from adding LV between 'ARP' and 'EUPC': {proposed_auc}")

Processing fold 0 using 7 cores takes 5.011972904205322 seconds
Processing fold 1 using 7 cores takes 4.623942852020264 seconds
Processing fold 2 using 7 cores takes 4.686042070388794 seconds
Processing fold 3 using 7 cores takes 4.5857250690460205 seconds
Processing fold 4 using 7 cores takes 4.645462989807129 seconds
AUC from adding LV between 'ARP' and 'EUPC': 0.2972222222222222
